# Notebook 2 — Crop 640×640 per-cell patches from yellow masks into all channels

Use the integer label mask for cropping. That means the uint16 label image you saved as _masks.tiff (preferred) or _masks.png. Overlays and binary previews are only for QA; do not drive cropping from them.

Below is a clean, robust cropper that:
	•	Prefers TIFF masks, falls back to PNG.
	•	Fixes the basename issue by swapping _yellow to each target channel.
	•	Lets you filter tiny labels.
	•	Crops fixed squares or tight bboxes with optional padding.
	•	Writes per-cell crops for all channels plus a CSV manifest with ROI metadata.

Notes
	•	The original output is untouched in analysis/cell_crops.
	•	The new PP output in analysis/cell_crops_pp uses the same filenames, so one-to-one pairing is guaranteed.
	•	The manifest has both original and PP paths so you can swap easily downstream.
	•	Tune context_px and background_mode to match your visualization or model needs.

In [4]:
# Notebook 2 — Crop 640×640 per-cell patches from yellow masks into all channels
# Outputs:
#   analysis/cell_crops/      -> original crops
#   analysis/cell_crops_pp/   -> neighbor-removed crops
#   analysis/cell_crops/path_list.csv
#   analysis/cell_crops_pp/path_list.csv
#
# Each path_list.csv has exactly 6 columns required by SubCellPortable:
# r_image,y_image,b_image,g_image,output_folder,output_prefix

from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any
import numpy as np
from skimage import io, measure
from skimage.transform import resize
from skimage.segmentation import expand_labels, find_boundaries
from skimage.morphology import label as cc_label
from skimage.color import gray2rgb
from tqdm import tqdm
import pandas as pd
import shutil
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# -------------------- Configuration --------------------
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Raw images are expected here: data/<channel>/*_<channel>.<ext>
img_root     = project_root / "data"

# Cellpose masks
masks_root   = project_root / "analysis" / "cellpose_results"
ref_channel  = "yellow"
ref_dir      = masks_root / ref_channel / "masks"

# Outputs
out_orig = project_root / "analysis" / "cell_crops"
out_pp   = project_root / "analysis" / "cell_crops_pp"

# Channels (order defines CSV columns)
channels: List[str] = ["red", "yellow", "blue", "green"]

# File prefs
prefer_tiff = True
image_exts  = [".jpg", ".jpeg", ".tif", ".tiff", ".png"]

# Cropping behavior (unchanged)
crop_size: Optional[int] = 640   # fixed square
min_area_px: int = 300           # drop tiny labels
max_cells_per_image: Optional[int] = None
require_all_channels: bool = False

# Post-processing (pp only)
# Non-overlapping context around the target; 0 keeps only label pixels
context_px: int = 12

# Background fill for masked-out neighbors
# Options: "zero", "percentile", "edge_mean"
background_mode: str = "zero"
background_percentile: float = 5.0

# Safeguards so pp tiles never go black
min_mask_area_frac: float = 0.003   # if target mask < 0.3% of tile, keep original crop
keep_only_largest_cc: bool = True

# Debug overlays (mask edge on yellow) for quick inspection
debug_overlays: bool = True
debug_max_overlays_per_image: int = 12

# SubCellPortable output folder value for each row (can be relative like 'output')
subcell_output_folder_value: str = "output"

# -------------------- Helpers --------------------
def ensure_fresh_output(root: Path, chans: List[str]):
    if root.exists():
        shutil.rmtree(root)
    for ch in chans:
        (root / ch).mkdir(parents=True, exist_ok=True)

def list_ref_masks(ref_dir: Path, prefer_tiff: bool=True) -> List[Path]:
    tifs = sorted(ref_dir.glob("*_masks.tif")) + sorted(ref_dir.glob("*_masks.tiff"))
    pngs = sorted(ref_dir.glob("*_masks.png"))
    if prefer_tiff and tifs:
        return tifs
    return tifs or pngs

def read_label_mask(path: Path) -> np.ndarray:
    m = io.imread(path)
    if m.ndim > 2:
        m = m[..., 0]
    return m

def to_labeled(mask: np.ndarray, min_area: int) -> np.ndarray:
    m = mask[..., 0] if mask.ndim > 2 else mask
    # already instance labels?
    if (m.dtype.kind in "iu") and (m.max() > 1):
        lab = m.astype(np.int32)
    else:
        from skimage.measure import label
        lab = label(m > 0, connectivity=1)
    # drop tiny objects
    if min_area > 0 and lab.max() > 0:
        keep = [r.label for r in measure.regionprops(lab) if r.area >= min_area]
        if keep:
            lab = measure.label(np.isin(lab, keep), connectivity=1).astype(np.int32)
        else:
            lab = np.zeros_like(lab, dtype=np.int32)
    return lab

def square_window(center_r: float, center_c: float, size: int, shape_hw: Tuple[int,int]) -> Tuple[int,int,int,int]:
    H, W = shape_hw
    half = size // 2
    r0 = int(round(center_r)) - half
    c0 = int(round(center_c)) - half
    r0 = max(r0, 0); c0 = max(c0, 0)
    r1 = min(r0 + size, H)
    c1 = min(c0 + size, W)
    if r1 - r0 < size: r0 = max(r1 - size, 0)
    if c1 - c0 < size: c0 = max(c1 - size, 0)
    return (r0, c0, r1, c1)

def safe_crop(img: np.ndarray, win: Tuple[int,int,int,int], pad_value=0) -> np.ndarray:
    r0, c0, r1, c1 = win
    H, W = img.shape[:2]
    sr0, sc0 = max(r0, 0), max(c0, 0)
    sr1, sc1 = min(r1, H), min(c1, W)
    out_h, out_w = r1 - r0, c1 - c0
    out_shape = (out_h, out_w) if img.ndim == 2 else (out_h, out_w, img.shape[2])
    out = np.full(out_shape, pad_value, dtype=img.dtype)
    dr, dc = sr0 - r0, sc0 - c0
    out[dr:dr + (sr1 - sr0), dc:dc + (sc1 - sc0)] = img[sr0:sr1, sc0:sc1]
    return out

def swap_suffix_yellow_to_channel(stem_with_channel: str, target_ch: str, ref_ch: str) -> str:
    if stem_with_channel.endswith(f"_{ref_ch}"):
        return stem_with_channel[:-(len(ref_ch)+1)] + f"_{target_ch}"
    return stem_with_channel

def read_image_for_channel(stem_yellow: str, ch: str, ch_dir: Path):
    target_stem = swap_suffix_yellow_to_channel(stem_yellow, ch, ref_channel)
    for ext in image_exts:
        p = ch_dir / f"{target_stem}{ext}"
        if p.exists():
            return io.imread(p), p
    return None, None

def resize_bool(mask_bool: np.ndarray, target_hw: Tuple[int,int]) -> np.ndarray:
    if mask_bool.shape != target_hw:
        m = resize(mask_bool.astype(np.uint8), target_hw,
                   order=0, preserve_range=True, anti_aliasing=False, mode="edge")
        return m > 0.5
    return mask_bool

def largest_cc(mask_bool: np.ndarray) -> np.ndarray:
    if not mask_bool.any():
        return mask_bool
    lbl = cc_label(mask_bool)
    vals, counts = np.unique(lbl[lbl > 0], return_counts=True)
    if len(vals) == 0:
        return np.zeros_like(mask_bool, dtype=bool)
    keep_val = vals[np.argmax(counts)]
    return lbl == keep_val

def apply_background(crop: np.ndarray, mask_bool: np.ndarray) -> np.ndarray:
    out = crop.copy()
    bg = ~mask_bool
    if crop.ndim == 2:
        if background_mode == "zero":
            out[bg] = 0
        elif background_mode == "percentile":
            out[bg] = np.percentile(crop, background_percentile)
        elif background_mode == "edge_mean":
            f = 10
            edges = np.concatenate([crop[:f, :].ravel(), crop[-f:, :].ravel(),
                                    crop[:, :f].ravel(), crop[:, -f:].ravel()])
            out[bg] = edges.mean() if edges.size else 0
        else:
            out[bg] = 0
    else:
        if background_mode == "zero":
            out[bg, :] = 0
        elif background_mode == "percentile":
            fill = np.percentile(crop, background_percentile)
            out[bg, :] = fill
        elif background_mode == "edge_mean":
            f = 10
            edges = np.concatenate([
                crop[:f, :, :].reshape(-1, crop.shape[-1]),
                crop[-f:, :, :].reshape(-1, crop.shape[-1]),
                crop[:, :f, :].reshape(-1, crop.shape[-1]),
                crop[:, -f:, :].reshape(-1, crop.shape[-1]),
            ], axis=0)
            out[bg, :] = edges.mean(axis=0) if edges.size else 0
        else:
            out[bg, :] = 0
    return out.astype(crop.dtype)

def overlay_mask(img2d: np.ndarray, mask_bool: np.ndarray) -> np.ndarray:
    if img2d.ndim == 3:
        base = img2d if img2d.shape[2] == 3 else np.repeat(img2d, 3, axis=2)
    else:
        base = gray2rgb(img2d)
    base = base.astype(np.float32)
    bnd = find_boundaries(mask_bool, mode="inner")
    base[..., 0] = np.maximum(base[..., 0], bnd.astype(np.float32) * (base.max() if base.max() > 0 else 1.0))
    if np.issubdtype(img2d.dtype, np.integer):
        base = np.clip(base, 0, np.iinfo(img2d.dtype).max)
        return base.astype(img2d.dtype)
    return np.clip(base, 0, 1.0)

def write_path_list(rows: List[List[str]], out_dir: Path, out_name: str = "path_list.csv"):
    """rows: [[r,y,b,g,output_folder,output_prefix], ...]"""
    df = pd.DataFrame(rows, columns=["r_image","y_image","b_image","g_image","output_folder","output_prefix"])
    out_csv = out_dir / out_name
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"Wrote {out_csv} with {len(df)} rows")

# -------------------- Run --------------------
# 1) locate masks
mask_files = list_ref_masks(ref_dir, prefer_tiff=prefer_tiff)
if not mask_files:
    raise FileNotFoundError(f"No *_masks.tif(f)/png found under {ref_dir}")

# 2) fresh outputs
ensure_fresh_output(out_orig, channels)
ensure_fresh_output(out_pp,   channels)
if debug_overlays:
    (out_pp / "_debug").mkdir(parents=True, exist_ok=True)

print(f"Using masks: {ref_dir}")
print(f"Found {len(mask_files)} mask file(s)")
print(f"Original crops -> {out_orig}")
print(f"PP crops       -> {out_pp}")

# CSV rows to build path_list.csv for each set
rows_orig: List[List[str]] = []
rows_pp:   List[List[str]] = []

for mpath in tqdm(mask_files, desc="Cropping"):
    stem_with_suffix = mpath.stem.replace("_masks", "")

    # load labels and regions
    lab_full = to_labeled(read_label_mask(mpath), min_area=min_area_px)
    regs = measure.regionprops(lab_full)
    if not regs:
        continue

    # cap per image if asked
    if max_cells_per_image is not None and len(regs) > max_cells_per_image:
        regs = sorted(regs, key=lambda r: r.area, reverse=True)[:max_cells_per_image]

    # preload channels
    raw_by_ch: Dict[str, Tuple[Optional[np.ndarray], Optional[Path]]] = {}
    for ch in channels:
        ch_dir = img_root / ch
        raw_by_ch[ch] = read_image_for_channel(stem_with_suffix, ch, ch_dir)

    if require_all_channels and any(raw_by_ch[ch][0] is None for ch in channels):
        continue

    dbg_written = 0

    for reg in regs:
        label_id = reg.label

        # fixed square window centered on the region
        win = square_window(reg.centroid[0], reg.centroid[1], crop_size, lab_full.shape)

        # label window for PP
        lab_win = safe_crop(lab_full, win)

        # target mask on window with non-overlapping context
        target_mask = (lab_win == label_id)
        if context_px > 0:
            lab_exp = expand_labels(lab_win, distance=context_px)  # non-overlapping growth
            target_mask = target_mask | (lab_exp == label_id)      # never shrink original label

        # per-channel outputs
        paths_o: Dict[str, str] = {}
        paths_p: Dict[str, str] = {}
        missing = False

        for ch in channels:
            img, img_path = raw_by_ch[ch]
            if img is None:
                missing = True
                continue

            # --- original crop ---
            crop = safe_crop(img, win)
            if crop.shape[0] != crop_size or crop.shape[1] != crop_size:
                crop = resize(crop, (crop_size, crop_size),
                              order=1, mode="edge", anti_aliasing=True, preserve_range=True).astype(img.dtype)
            p_o = out_orig / ch / f"{stem_with_suffix}_cell{label_id}.png"
            io.imsave(p_o, crop, check_contrast=False)
            paths_o[ch] = str(p_o)

            # --- pp crop (neighbors removed) ---
            m_bool = resize_bool(target_mask, crop.shape[:2])
            if keep_only_largest_cc:
                m_bool = largest_cc(m_bool)

            # safety: fallback to original if mask too small or empty
            mask_area = int(m_bool.sum())
            if mask_area == 0 or mask_area / float(crop_size * crop_size) < min_mask_area_frac:
                crop_pp = crop
            else:
                crop_pp = apply_background(crop, m_bool)

            p_p = out_pp / ch / f"{stem_with_suffix}_cell{label_id}.png"
            io.imsave(p_p, crop_pp, check_contrast=False)
            paths_p[ch] = str(p_p)

            # optional debug overlay on yellow only
            if debug_overlays and ch == "yellow" and dbg_written < debug_max_overlays_per_image:
                ov = overlay_mask(crop, m_bool)
                io.imsave(out_pp / "_debug" / f"{stem_with_suffix}_cell{label_id}_overlay.png", ov, check_contrast=False)
                dbg_written += 1

        if require_all_channels and missing:
            # remove partial outputs
            for p in paths_o.values(): Path(p).unlink(missing_ok=True)
            for p in paths_p.values(): Path(p).unlink(missing_ok=True)
            continue

        # --- build SubCell rows (exactly 6 columns) ---
        # output_prefix: image ID + cell ID (unique and informative)
        out_prefix = f"{stem_with_suffix}_cell{label_id}"

        rows_orig.append([
            paths_o.get("red",""),
            paths_o.get("yellow",""),
            paths_o.get("blue",""),
            paths_o.get("green",""),
            subcell_output_folder_value,
            out_prefix
        ])

        rows_pp.append([
            paths_p.get("red",""),
            paths_p.get("yellow",""),
            paths_p.get("blue",""),
            paths_p.get("green",""),
            subcell_output_folder_value,
            out_prefix
        ])

# -------------------- Write path_list.csv files --------------------
write_path_list(rows_orig, out_orig)     # analysis/cell_crops/path_list.csv
write_path_list(rows_pp,   out_pp)       # analysis/cell_crops_pp/path_list.csv

# -------------------- Sanity checks --------------------
counts_o  = {ch: len(list((out_orig / ch).glob("*.png"))) for ch in channels}
counts_pp = {ch: len(list((out_pp   / ch).glob("*.png"))) for ch in channels}
print("Original counts per channel:    ", counts_o)
print("Post-processed counts per channel:", counts_pp)

ok = (
    len(set(counts_o.values())) == 1
    and len(set(counts_pp.values())) == 1
    and list(counts_o.values())[0] == list(counts_pp.values())[0]
    and len(rows_orig) == len(rows_pp)
)
print("1-to-1 identity of tiles and CSV rows:" , "OK" if ok else "MISMATCH")

Using masks: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results/yellow/masks
Found 10 mask file(s)
Original crops -> /Users/ashi/github/cm4ai_codefest2025/analysis/cell_crops
PP crops       -> /Users/ashi/github/cm4ai_codefest2025/analysis/cell_crops_pp


Cropping: 100%|██████████| 10/10 [00:26<00:00,  2.61s/it]

Wrote /Users/ashi/github/cm4ai_codefest2025/analysis/cell_crops/path_list.csv with 66 rows
Wrote /Users/ashi/github/cm4ai_codefest2025/analysis/cell_crops_pp/path_list.csv with 66 rows
Original counts per channel:     {'red': 66, 'yellow': 66, 'blue': 66, 'green': 66}
Post-processed counts per channel: {'red': 66, 'yellow': 66, 'blue': 66, 'green': 66}
1-to-1 identity of tiles and CSV rows: OK
